# 00 — Dataset Audit

**CPU only. Run this first. It costs no GPU quota and it is the evidence base for everything else.**

Three questions, answered from the data rather than asserted:

1. Do filenames carry patient identifiers? *(the original thesis said yes)*
2. Does the slice ordering carry scan structure, i.e. is the block proxy defensible?
3. How much grouping does each split strategy actually break?

Outputs `results/audit.json` and two figures used in the paper.

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

## 1. Registry and block construction

In [ ]:
recs = splits.build_registry(ROOT)
assert len(recs) == 1097, len(recs)
assert len({r["block_id"] for r in recs}) == 110

print(f"images: {len(recs)}   proxy blocks: {len({r['block_id'] for r in recs})}")
print(json.dumps(audit.block_size_summary(recs), indent=1))

## 2. Do filenames encode patients?

The thesis states that a "case/group identifier was used as the prefix of the file name that was
shared by the successive slices". This is the check that claim never received.

In [ ]:
fa = audit.filename_audit(recs)
for k, v in fa.items():
    if k.startswith("_"):
        continue
    print(f"{k}")
    print(f"   images                    {v['n_images']}")
    print(f"   distinct filename prefixes {v['n_distinct_prefixes']}  -> {v['prefixes']}")
    print(f"   distinct (n) values        {v['n_distinct_slice_indices']}"
          f"   equals image count: {v['distinct_indices_equals_n_images']}")
print()
print(fa["_conclusion"])

In [ ]:
# Lexicographic sort also destroys numeric order -- the second half of the original bug.
print("sorted(os.listdir(...)) order, first 10 Benign files:")
for f in audit.lexicographic_scramble(recs, "Bengin cases", 10):
    print("  ", f)

## 3. Is the slice ordering informative?

If neighbouring slice indices are far more similar than distant ones, the ordering carries scan
structure and contiguous blocks are a defensible patient proxy. **The distance should rise with
lag and plateau at roughly the published slices-per-patient for that class** (Benign 8,
Malignant 14, Normal 7.6). If it is flat, the proxy has no basis and we report that instead.

In [ ]:
oa = audit.ordering_audit(recs, max_lag=40)
for c, v in oa.items():
    print(f"{c:<18} lag1={v['mean_distance_lag1']:6.3f}  far={v['mean_distance_far']:6.3f}"
          f"  ratio={v['ratio_lag1_to_far']}")

In [ ]:
import matplotlib.pyplot as plt
BLOCK = {"Bengin cases": 8.0, "Malignant cases": 14.0, "Normal cases": 7.6}

fig, ax = plt.subplots(figsize=(7, 4.2))
for c, v in oa.items():
    lags = sorted(v["distance_by_lag"])
    ax.plot(lags, [v["distance_by_lag"][l] for l in lags], marker="o", ms=3,
            label=f"{c.replace(' cases','')} (block={BLOCK[c]:.1f})")
    ax.axhline(v["mean_distance_far"], ls=":", lw=0.8, color="grey")
    ax.axvline(BLOCK[c], ls="--", lw=0.8, alpha=0.4)
ax.set_xlabel("gap in slice index (lag)")
ax.set_ylabel("mean perceptual-hash distance")
ax.set_title("Slice-sequence autocorrelation vs. published slices-per-patient")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{RESULTS}/fig_ordering_autocorrelation.png", dpi=200)
plt.show()

## 4. Are near-duplicates concentrated inside blocks?

`lift` is how many times more likely a near-duplicate pair is to share a block than a random pair
is. A large lift means blocks capture real within-scan correlation.

In [ ]:
da = audit.duplicate_audit(recs, threshold=6)
print(f"{'class':<18}{'near-dup pairs':>15}{'% same block':>14}{'base rate':>11}{'lift':>7}")
for c, v in da.items():
    print(f"{c:<18}{v['n_near_duplicate_pairs']:>15}{v['pct_near_dup_pairs_same_block']:>14}"
          f"{v['pct_all_pairs_same_block']:>11}{v['lift']:>7}")

## 5. How much grouping does each split strategy break?

`blocks_straddling_partitions` is the number of proxy patients with slices in more than one
partition. **This is the table the original thesis printed without computing.**

In [ ]:
rows = []
for strat in splits.STRATEGIES:
    rep = splits.verify_split(splits.make_split(recs, strat, seed=42))
    rows.append((strat, rep["blocks_straddling_partitions"], rep["blocks_are_disjoint"],
                 rep["n_images"]))
    print(splits.split_table(recs)); print("-" * 60)

print(f"\n{'strategy':<20}{'straddling':>12}{'disjoint':>11}")
for s, n, d, _ in rows:
    print(f"{s:<20}{n:>12}{str(d):>11}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
names = [r[0] for r in rows]; vals = [r[1] for r in rows]
cols = ["#c0392b" if v > 0 else "#27ae60" for v in vals]
ax.barh(names, vals, color=cols)
for i, v in enumerate(vals):
    ax.text(v + 1, i, str(v), va="center")
ax.set_xlabel("proxy patients with slices in >1 partition (of 110)")
ax.set_title("Grouping broken by each split strategy")
fig.tight_layout(); fig.savefig(f"{RESULTS}/fig_straddling_blocks.png", dpi=200); plt.show()

## 6. Recovering scan boundaries — changepoint detection

Equal-sized blocks assume every scan contributed the same number of slices, which is false. Real
scan boundaries should appear as **discontinuities** in the adjacent-slice distance: slices within
a scan are near-continuous, the jump from one scan to the next is not.

Given the published subject count `K`, take the top `K-1` jumps as boundaries. **`prominence_ratio`
is the test**: mean jump at the selected boundaries over mean jump elsewhere. A ratio near 1 means
no boundary structure exists and the method has failed.

In [ ]:
from iqoth import external
cp = {}
print(f"{'class':<18}{'blocks':>8}{'min':>6}{'max':>6}{'mean':>7}{'prominence':>12}")
for cname, k in splits.PATIENTS_PER_CLASS.items():
    sub = [r for r in recs if r["class_name"] == cname]
    info = external.refine_blocks_by_changepoint(sub, k)
    cp[cname] = info
    b = info["block_sizes"]
    print(f"{cname:<18}{b['n_blocks']:>8}{b['min']:>6}{b['max']:>6}{b['mean']:>7}"
          f"{info['prominence_ratio']:>11.2f}x")
print("\nUnequal block sizes are expected and are a point in this method's favour:")
print("real scans do not contribute equal numbers of slices.")

### Methods that did NOT work

Reported so the attempts are documented rather than quietly dropped. Both are negative results
that belong in the paper.

In [ ]:
# (a) Silhouette sweep for block size -- fails: the sequence is a continuous chain with no
#     gaps between adjacent groups, so silhouette rewards small blocks that capture local
#     smoothness rather than scan boundaries.
published = {"Bengin cases": 8.0, "Malignant cases": 14.0, "Normal cases": 7.6}
sil = {}
for cname in splits.PATIENTS_PER_CLASS:
    sub = [r for r in recs if r["class_name"] == cname]
    e = external.estimate_block_size(sub, candidates=range(2, 41))
    sil[cname] = e
    print(f"{cname:<18} silhouette estimate {e['best_block_size']:>3}   "
          f"published {published[cname]:>5}   best_sil={e['best_silhouette']:.4f}")
print("-> silhouette does NOT recover the published block sizes. Reported as a negative result.")

In [ ]:
# (b) Unsupervised boundary COUNT by robust z-score -- over-detects, which is unsafe:
#     a scan fragmented into two detected groups can still straddle partitions.
for cname, k in splits.PATIENTS_PER_CLASS.items():
    sub = [r for r in recs if r["class_name"] == cname]
    for z in (2.0, 2.5, 3.0):
        est = external.estimate_n_groups_unsupervised(sub, z=z)
        print(f"{cname:<18} z={z}  detected K={est:>3}  true K={k:>3}  err={est-k:+d}")
print("-> the published subject count is still required. State this plainly in the paper.")

## 7. Split integrity under BOTH proxies

The equal-cut and changepoint proxies must both be reported — agreement between them is the
robustness check.

In [ ]:
integrity = {}
for proxy in ("equal_cut", "changepoint"):
    fresh = splits.build_registry(ROOT)
    if proxy == "changepoint":
        for cname, k in splits.PATIENTS_PER_CLASS.items():
            external.refine_blocks_by_changepoint(
                [r for r in fresh if r["class_name"] == cname], k)
    integrity[proxy] = {}
    for strat in splits.STRATEGIES:
        rep = splits.verify_split(splits.make_split(fresh, strat, seed=42))
        integrity[proxy][strat] = rep["blocks_straddling_partitions"]

print(f"{'strategy':<20}{'equal-cut':>12}{'changepoint':>14}")
for strat in splits.STRATEGIES:
    print(f"{strat:<20}{integrity['equal_cut'][strat]:>12}{integrity['changepoint'][strat]:>14}")
print("\n(blocks straddling partitions, out of 110)")

In [ ]:
out = {"filename_audit": fa, "ordering_audit": oa, "duplicate_audit": da,
       "block_summary": audit.block_size_summary(recs),
       "straddling_by_strategy": {s: n for s, n, _, _ in rows},
       "changepoint": {k: {kk: vv for kk, vv in v.items() if kk != "boundary_positions"}
                       for k, v in cp.items()},
       "silhouette_negative_result": {k: {"best_block_size": v["best_block_size"],
                                          "best_silhouette": v["best_silhouette"]}
                                      for k, v in sil.items()},
       "split_integrity_both_proxies": integrity}
with open(f"{RESULTS}/audit.json", "w") as f:
    json.dump(out, f, indent=2)
print("saved", f"{RESULTS}/audit.json")